In [53]:
from langchain_huggingface import HuggingFaceEndpoint
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.pydantic_v1 import BaseModel, Field
from typing import List, Tuple
from backend.components.prompts import Prompts
from backend.components.retriever import Retriever
from langchain_chroma import Chroma
from langchain.chains.history_aware_retriever import create_history_aware_retriever
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chains.retrieval import create_retrieval_chain
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

In [51]:
callbacks = [StreamingStdOutCallbackHandler()]

llm = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-Nemo-Instruct-2407",
    task="text-generation",
    max_new_tokens=1024,
    temperature=0.01,
    callbacks=callbacks,
    streaming=True,
    stop_sequences=["<unk>", "</s>", "}"],
)

In [52]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectordb = Chroma(
    persist_directory="backend/data/db", embedding_function=embeddings
)

retriever = Retriever(vectordb=vectordb, k=4, threshold=1.0)

prompts = Prompts()

# chain that stuffs documents into context
combine_docs_chain = create_stuff_documents_chain(
    llm=llm, prompt=prompts.qa_chat_prompt_template
)

history_aware_retriever = create_history_aware_retriever(
    llm=llm,
    retriever=retriever,
    prompt=prompts.rephrase_chat_prompt_template,
)

/Users/jessedingley/Documents/rag/broke_env/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [54]:
qa = create_retrieval_chain(
    retriever=history_aware_retriever, combine_docs_chain=combine_docs_chain
)

In [56]:
query = "Give e a 4-week itinerary for Nepal."

In [58]:
chat_history = [
    ("assistant", f"Hey there! Alma here, how can I help you?")
]

In [59]:
for chunk in qa.stream(
    input={"input": query + " ASSISTANT: ", "chat_history": chat_history}
):
    pass

4-Week Itinerary for Nepal

1. **Kathmandu (Week 1)**
   - Day 1-3: Explore Kathmandu Valley - Kathmandu, Bhaktapur, and Patan. Visit UNESCO World Heritage Sites like Kathmandu Durbar Square, Swayambhunath, and Pashupatinath.
   - Day 4-7: Trek to Nagarkot and Changunarayan. Enjoy panoramic views of the Himalayas and visit the ancient temple of Changunarayan.

2. **Pokhara (Week 2)**
   - Day 8-10: Travel to Pokhara and explore the city. Visit the International Mountain Museum, Phewa Lake, and the Peace Pagoda.
   - Day 11-14: Trek the Annapurna Base Camp (ABC) route. Enjoy the stunning views of the Annapurna range and the diverse landscapes of the Annapurna Conservation Area.

3. **Chitwan (Week 3)**
   - Day 15-17: Travel to Chitwan National Park. Enjoy jungle safaris, canoe rides, and cultural programs. Spot wildlife like rhinos, deer, and various bird species.

4. **Lumbini (Week 4)**
   - Day 18-21: Travel to Lumbini, the birthplace of Gautam Buddha. Visit the Maya Devi Temple, As

ValueError: not enough values to unpack (expected 2, got 0)

In [38]:

"""
formatting that worked:


 som tam\n* **Kanchanaburi** (1-2 days)\n\t+ Death Rai

" **Week 1: Bangkok to Chiang Mai** - Here's a list of places to visit during your first week in Northern Thailand:\n\n* **Bangkok** (2-3 days)\n\t+ Temples: Wat Arun, Wat Pho, Wat Traimit\n\t+ Markets: Chatuchak Weekend Market, Floating Markets (Damnoen Saduak or Amphawa)\n\t+ Street Food: Try pad thai, mango sticky rice, and som tam\n* **Kanchanaburi** (1-2 days)\n\t+ Death Railway and Bridge on the River Kwai\n\t+ Hellfire Pass Memorial and Museum\n\t+ Erawan National Park (if time allows)\n* **Ayutthaya** (1 day)\n\t+ Ancient ruins: Wat Chaiwatthanaram, Wat Phra Sri Sanphet, Wat Mahathat\n\t+ Bang Pa-In Summer Palace (if time allows)\n* **Chiang Mai** (2-3 days)\n\t+ Temples: Wat Phra That Doi Suthep, Wat Chedi Luang, Wat Suan Dok\n\t+ Markets: Sunday Walking Street, Night Bazaar\n\t+ Cafes: Try Chiang Mai's famous coffee and desserts"
"""

'\nformatting that worked:\n\n\n som tam\n* **Kanchanaburi** (1-2 days)\n\t+ Death Rai\n\n" **Week 1: Bangkok to Chiang Mai** - Here\'s a list of places to visit during your first week in Northern Thailand:\n\n* **Bangkok** (2-3 days)\n\t+ Temples: Wat Arun, Wat Pho, Wat Traimit\n\t+ Markets: Chatuchak Weekend Market, Floating Markets (Damnoen Saduak or Amphawa)\n\t+ Street Food: Try pad thai, mango sticky rice, and som tam\n* **Kanchanaburi** (1-2 days)\n\t+ Death Railway and Bridge on the River Kwai\n\t+ Hellfire Pass Memorial and Museum\n\t+ Erawan National Park (if time allows)\n* **Ayutthaya** (1 day)\n\t+ Ancient ruins: Wat Chaiwatthanaram, Wat Phra Sri Sanphet, Wat Mahathat\n\t+ Bang Pa-In Summer Palace (if time allows)\n* **Chiang Mai** (2-3 days)\n\t+ Temples: Wat Phra That Doi Suthep, Wat Chedi Luang, Wat Suan Dok\n\t+ Markets: Sunday Walking Street, Night Bazaar\n\t+ Cafes: Try Chiang Mai\'s famous coffee and desserts"\n'